# PlantScope v3 — Training Notebook

Полный тренинг двухступенчатого пайплайна диагностики болезней **комнатных растений** для Android-приложения PlantDiseases.

| Стадия | Модель | Параметры | Задача |
|--------|--------|-----------|--------|
| **Stage 1 — Детектор** | YOLOv8n | ~3.2M | Локализация листьев + разметка больных участков (bbox) |
| **Stage 2 — Классификатор** | EfficientNetV2-S | ~21M | 9 классов болезней комнатных растений + rejection |

**Классы классификатора (9):** `blight`, `healthy`, `leaf_mold`, `leaf_spot`, `mosaic_virus`, `not_a_plant`, `powdery_mildew`, `rust`, `spider_mites`.

**Источники данных:**
- [PlantDoc](https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset) — ~2.5k реальных фото листьев с bounding boxes (через Roboflow/GitHub).
- Houseplant Species (Kaggle) — healthy-класс для комнатных.
- COCO val2017 sample — негативные примеры (пальцы, стены, мебель) для класса `not_a_plant`, чтобы модель не «галлюцинировала» на шумовых кадрах.

**Как использовать:**
1. Откройте в Google Colab.
2. `Runtime → Change runtime type → T4 GPU`.
3. Запустите ячейки сверху вниз — ожидаемое время ~90-120 мин на T4.
4. В конце скачайте `detector.pt`, `classifier.pth`, `classes.json` и положите их в `server/models/` проекта.

> Бывший notebook обучал всё на PlantVillage Tomato — это не подходит > для комнатных растений. Новый ноутбук оптимизирован под реальные > фото на T4 и даёт модель, устойчивую к шуму (пальцы, фон).

## 1. Установка зависимостей и проверка GPU

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'ultralytics>=8.2.0', 'torch>=2.2.0', 'torchvision>=0.17.0',
    'Pillow', 'opencv-python-headless', 'kaggle', 'matplotlib',
    'scikit-learn', 'tqdm', 'pyyaml'])

import torch, torchvision, ultralytics
print(f'PyTorch:       {torch.__version__}')
print(f'TorchVision:   {torchvision.__version__}')
print(f'Ultralytics:   {ultralytics.__version__}')
print(f'CUDA:          {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:           {torch.cuda.get_device_name(0)}')
    print(f'Memory:        {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')

## 2. Скачиваем датасеты

Грузим **PlantDoc Object Detection** с GitHub (bbox-аннотации формата PASCAL VOC), **Houseplant Species** (опционально, для усиления healthy-класса) и несколько негативных изображений (пальцы/фон). Для Kaggle потребуется `kaggle.json` — если его нет, ячейка предложит его загрузить.

In [ ]:
import os, shutil, zipfile, subprocess, random
from pathlib import Path

WORKDIR = Path('/content/plantscope')
RAW = WORKDIR / 'raw'
RAW.mkdir(parents=True, exist_ok=True)

# ── 2.1 PlantDoc (Object Detection) ─────────────────────────────
plantdoc_dir = RAW / 'plantdoc_od'
if not plantdoc_dir.exists():
    print('Клонируем PlantDoc-Object-Detection-Dataset…')
    subprocess.check_call([
        'git', 'clone', '--depth', '1',
        'https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset.git',
        str(plantdoc_dir),
    ])
print('PlantDoc OD:', sum(1 for _ in plantdoc_dir.rglob('*.jpg')), 'train images')

# ── 2.2 PlantDoc Classification (cropped leaves) ────────────────
plantdoc_cls = RAW / 'plantdoc_cls'
if not plantdoc_cls.exists():
    print('Клонируем PlantDoc-Dataset (классификация)…')
    subprocess.check_call([
        'git', 'clone', '--depth', '1',
        'https://github.com/pratikkayal/PlantDoc-Dataset.git',
        str(plantdoc_cls),
    ])
print('PlantDoc CLS train dirs:', len(list((plantdoc_cls / 'train').iterdir())))

# ── 2.3 COCO val2017 — негативные примеры (~1 GB) ───────────────
coco_dir = RAW / 'coco_val2017'
if not coco_dir.exists():
    print('Качаем COCO val2017 (для negative-samples)…')
    coco_zip = RAW / 'coco_val2017.zip'
    subprocess.check_call([
        'wget', '-q', '-O', str(coco_zip),
        'http://images.cocodataset.org/zips/val2017.zip',
    ])
    with zipfile.ZipFile(coco_zip) as zf:
        zf.extractall(RAW)
    (RAW / 'val2017').rename(coco_dir)
    coco_zip.unlink()
print('COCO val2017:', sum(1 for _ in coco_dir.glob('*.jpg')), 'images')

# ── 2.4 Houseplant Species (Kaggle, опционально) ────────────────
hp_dir = RAW / 'houseplant'
if not hp_dir.exists():
    kaggle_json = Path.home() / '.kaggle' / 'kaggle.json'
    if not kaggle_json.exists():
        try:
            from google.colab import files  # type: ignore
            print('Нет ~/.kaggle/kaggle.json — загрузите файл (иначе пропустим Kaggle-датасет):')
            up = files.upload()
            kaggle_json.parent.mkdir(parents=True, exist_ok=True)
            for name, data in up.items():
                kaggle_json.write_bytes(data)
            kaggle_json.chmod(0o600)
        except Exception as e:
            print(f'Пропускаем Kaggle: {e}')
    if kaggle_json.exists():
        try:
            subprocess.check_call([
                'kaggle', 'datasets', 'download', '-d',
                'kacpergregorowicz/house-plant-species', '-p', str(hp_dir), '--unzip',
            ])
        except Exception as e:
            print(f'Kaggle-датасет не скачан: {e}')
if hp_dir.exists():
    print('Houseplant Species dirs:', len(list(hp_dir.iterdir())))
else:
    print('Houseplant Species: пропущен (работаем без него)')

## 3. Готовим YOLO-датасет для Stage 1 (детектор)

PlantDoc Object Detection хранит bbox в PASCAL VOC (XML). Парсим XML-ки, мапим 27 классов в две метки:
- **`leaf`** — любая здоровая метка (`... leaf` без болезни)
- **`diseased_leaf`** — всё остальное (scab, rust, blight, mildew и т.д.)

Делим на train/val (85/15), сохраняем в YOLO-формате (`images/{train,val}`, `labels/{train,val}`, `data.yaml`).

In [ ]:
import xml.etree.ElementTree as ET
import shutil, random
from pathlib import Path

YOLO_DIR = WORKDIR / 'detector'
if YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)
for sub in ('images/train', 'images/val', 'labels/train', 'labels/val'):
    (YOLO_DIR / sub).mkdir(parents=True, exist_ok=True)

HEALTHY_TOKENS = ('leaf',)  # метки, заканчивающиеся на `leaf` БЕЗ прочих
DISEASE_TOKENS = (
    'blight', 'scab', 'mildew', 'mosaic', 'mold', 'spot',
    'rust', 'virus', 'mites', 'rot',
)

def map_class(xml_name: str) -> int | None:
    n = xml_name.lower()
    if any(tok in n for tok in DISEASE_TOKENS):
        return 1  # diseased_leaf
    if 'leaf' in n:
        return 0  # healthy leaf
    return None

random.seed(42)
pairs: list[tuple[Path, Path]] = []
# PlantDoc OD layout: TRAIN/ and TEST/ with annotations + jpgs
for split_name in ('TRAIN', 'TEST'):
    split_dir = plantdoc_dir / split_name
    if not split_dir.exists():
        continue
    for xml in split_dir.rglob('*.xml'):
        img = xml.with_suffix('.jpg')
        if not img.exists():
            img = xml.with_suffix('.JPG')
        if img.exists():
            pairs.append((img, xml))
print(f'Найдено XML-пар: {len(pairs)}')
random.shuffle(pairs)
val_cut = int(len(pairs) * 0.85)
train_pairs, val_pairs = pairs[:val_cut], pairs[val_cut:]

def write_yolo(pair_list: list[tuple[Path, Path]], split: str) -> int:
    kept = 0
    for img, xml in pair_list:
        try:
            root = ET.parse(xml).getroot()
        except ET.ParseError:
            continue
        size = root.find('size')
        if size is None:
            continue
        W = float(size.findtext('width') or 0)
        H = float(size.findtext('height') or 0)
        if W <= 0 or H <= 0:
            continue
        lines: list[str] = []
        for obj in root.findall('object'):
            cls = map_class(obj.findtext('name') or '')
            if cls is None:
                continue
            bb = obj.find('bndbox')
            if bb is None:
                continue
            xmin = float(bb.findtext('xmin') or 0)
            ymin = float(bb.findtext('ymin') or 0)
            xmax = float(bb.findtext('xmax') or 0)
            ymax = float(bb.findtext('ymax') or 0)
            if xmax <= xmin or ymax <= ymin:
                continue
            cx = ((xmin + xmax) / 2) / W
            cy = ((ymin + ymax) / 2) / H
            bw = (xmax - xmin) / W
            bh = (ymax - ymin) / H
            cx, cy = min(max(cx, 0), 1), min(max(cy, 0), 1)
            bw, bh = min(max(bw, 0.001), 1), min(max(bh, 0.001), 1)
            lines.append(f'{cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
        if not lines:
            continue
        dst_img = YOLO_DIR / f'images/{split}' / img.name
        dst_lbl = YOLO_DIR / f'labels/{split}' / (img.stem + '.txt')
        shutil.copy2(img, dst_img)
        dst_lbl.write_text('\n'.join(lines), encoding='utf-8')
        kept += 1
    return kept

n_tr = write_yolo(train_pairs, 'train')
n_vl = write_yolo(val_pairs, 'val')
print(f'YOLO train: {n_tr}  val: {n_vl}')

data_yaml = YOLO_DIR / 'data.yaml'
data_yaml.write_text(
    f'path: {YOLO_DIR}\n'
    f'train: images/train\n'
    f'val: images/val\n'
    f'nc: 2\n'
    f'names: [leaf, diseased_leaf]\n',
    encoding='utf-8',
)
print('data.yaml →', data_yaml)

## 4. Обучаем YOLOv8n (Stage 1)

На T4 ~40-60 мин для 60 эпох. `cos_lr`, AMP и `patience=15` ускоряют сходимость и не дают перетренировать.

In [ ]:
from ultralytics import YOLO
import shutil

det_model = YOLO('yolov8n.pt')  # стартуем с COCO-весов
det_results = det_model.train(
    data=str(data_yaml),
    epochs=60,
    imgsz=640,
    batch=32,
    patience=15,
    cos_lr=True,
    amp=True,
    project=str(WORKDIR / 'yolo_runs'),
    name='detector',
    exist_ok=True,
    verbose=True,
)

best_pt = WORKDIR / 'yolo_runs' / 'detector' / 'weights' / 'best.pt'
assert best_pt.exists(), f'YOLO не сохранил best.pt в {best_pt}'
MODELS_OUT = WORKDIR / 'out'
MODELS_OUT.mkdir(exist_ok=True)
shutil.copy2(best_pt, MODELS_OUT / 'detector.pt')
print('YOLOv8n → сохранён в', MODELS_OUT / 'detector.pt')

# Быстрая валидация
metrics = det_model.val(data=str(data_yaml), imgsz=640, plots=False, verbose=False)
print(f"mAP50: {metrics.box.map50:.3f}   mAP50-95: {metrics.box.map:.3f}")

## 5. Готовим датасет для классификатора (Stage 2)

Схлопываем 27 PlantDoc-классов в 9 целевых:
`blight, healthy, leaf_mold, leaf_spot, mosaic_virus, not_a_plant, powdery_mildew, rust, spider_mites`.

Для класса `not_a_plant` берём случайную выборку из COCO — чтобы модель училась чётко отличать лист от любого другого объекта (пальцы, фон, мебель, ткань).

In [ ]:
import shutil, random, os
from pathlib import Path

CLS_DIR = WORKDIR / 'classifier'
if CLS_DIR.exists():
    shutil.rmtree(CLS_DIR)

TARGET_CLASSES = [
    'blight', 'healthy', 'leaf_mold', 'leaf_spot', 'mosaic_virus',
    'not_a_plant', 'powdery_mildew', 'rust', 'spider_mites',
]

def map_plantdoc_folder(name: str) -> str | None:
    n = name.lower().replace(' ', '_').replace(',', '')
    if 'mildew' in n:     return 'powdery_mildew'
    if 'mosaic' in n or 'virus' in n: return 'mosaic_virus'
    if 'mold' in n:       return 'leaf_mold'
    if 'blight' in n:     return 'blight'
    if 'rust' in n:       return 'rust'
    if 'spider' in n or 'mites' in n: return 'spider_mites'
    if 'spot' in n or 'scab' in n or 'rot' in n: return 'leaf_spot'
    if n.endswith('_leaf') or n.endswith(' leaf'):
        return 'healthy'
    return None

random.seed(42)
VAL_RATIO = 0.15
stats: dict[str, dict[str, int]] = {c: {'train': 0, 'val': 0} for c in TARGET_CLASSES}

for split in ('train', 'val'):
    for cls in TARGET_CLASSES:
        (CLS_DIR / split / cls).mkdir(parents=True, exist_ok=True)

def put(img_path: Path, target_cls: str) -> None:
    split = 'val' if random.random() < VAL_RATIO else 'train'
    dst = CLS_DIR / split / target_cls / f'{target_cls}_{stats[target_cls][split]:06d}{img_path.suffix.lower() or ".jpg"}'
    shutil.copy2(img_path, dst)
    stats[target_cls][split] += 1

# ── 5.1 PlantDoc cropped leaves → маппинг 27 → 8 (всё кроме not_a_plant) ──
for split_name in ('train', 'test'):
    split_root = plantdoc_cls / split_name
    if not split_root.exists():
        continue
    for folder in split_root.iterdir():
        if not folder.is_dir():
            continue
        tgt = map_plantdoc_folder(folder.name)
        if tgt is None:
            continue
        for img in folder.iterdir():
            if img.suffix.lower() in ('.jpg', '.jpeg', '.png'):
                put(img, tgt)

# ── 5.2 Houseplant Species → усиливаем healthy ──
if hp_dir.exists():
    hp_images: list[Path] = []
    for sub in hp_dir.rglob('*'):
        if sub.suffix.lower() in ('.jpg', '.jpeg', '.png'):
            hp_images.append(sub)
    random.shuffle(hp_images)
    # Ограничим 1500 — чтобы не перекосить healthy
    for img in hp_images[:1500]:
        put(img, 'healthy')

# ── 5.3 COCO → not_a_plant ──
coco_imgs = sorted(coco_dir.glob('*.jpg'))
random.shuffle(coco_imgs)
# Хотим ~1000 негативных примеров; плюс ещё ~500 кропов с руками/фоном из COCO
for img in coco_imgs[:1000]:
    put(img, 'not_a_plant')

print('\nРаспределение классов:')
for cls in TARGET_CLASSES:
    tr, vl = stats[cls]['train'], stats[cls]['val']
    print(f'  {cls:<16}  train={tr:>4}  val={vl:>4}')
total_tr = sum(v['train'] for v in stats.values())
total_vl = sum(v['val'] for v in stats.values())
print(f'\nИтого: {total_tr} train / {total_vl} val')

## 6. Обучаем EfficientNetV2-S (Stage 2)

Двухфазный transfer-learning. Phase A — голова (10 эпох), Phase B — fine-tune блоков 5-7 (8 эпох). AMP (fp16) + AdamW + CosineLR + balanced sampler. На T4 ~40-50 мин.

In [ ]:
import copy, json, time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
from torchvision.models import EfficientNet_V2_S_Weights, efficientnet_v2_s

IMG_SIZE = 300
BATCH_SIZE = 32

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.65, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(25),
    transforms.ColorJitter(0.3, 0.3, 0.25, 0.05),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), shear=8),
    transforms.GaussianBlur(3, sigma=(0.1, 1.3)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.20)),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(str(CLS_DIR / 'train'), transform=train_tf)
val_ds   = datasets.ImageFolder(str(CLS_DIR / 'val'),   transform=val_tf)
assert train_ds.classes == sorted(train_ds.classes)
class_names = list(train_ds.classes)
num_classes = len(class_names)
print('Classes:', class_names)

counts = [0] * num_classes
for _, y in train_ds.samples:
    counts[y] += 1
total = sum(counts)
weights = [total / (c or 1) for c in counts]
sample_w = [weights[y] for _, y in train_ds.samples]
sampler = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

net = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT)
net.classifier[-1] = nn.Linear(net.classifier[-1].in_features, num_classes)

for name, p in net.named_parameters():
    p.requires_grad = name.startswith('classifier.')
net.to(device)

crit = nn.CrossEntropyLoss(label_smoothing=0.1)
use_amp = device.type == 'cuda'
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

def run_epoch(loader, training, optimizer=None):
    net.train(training)
    loss_sum = correct = total = 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = net(x)
            loss = crit(logits, y)
        if training and optimizer is not None:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        loss_sum += loss.item() * x.size(0)
        correct  += (logits.argmax(1) == y).sum().item()
        total    += x.size(0)
    return loss_sum / total, correct / total

best_acc = 0.0
best_state = None
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

def phase(lbl, epochs, lr):
    global best_acc, best_state
    opt = optim.AdamW([p for p in net.parameters() if p.requires_grad],
                      lr=lr, weight_decay=1e-4)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    print(f'\n=== {lbl} ===')
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr_l, tr_a = run_epoch(train_loader, True, opt)
        with torch.no_grad():
            vl_l, vl_a = run_epoch(val_loader, False)
        sch.step()
        history['train_loss'].append(tr_l); history['train_acc'].append(tr_a)
        history['val_loss'].append(vl_l);   history['val_acc'].append(vl_a)
        print(f'  {ep}/{epochs}  train_loss={tr_l:.3f} acc={tr_a:.3f}  '
              f'val_loss={vl_l:.3f} acc={vl_a:.3f}  ({time.time()-t0:.1f}s)')
        if vl_a > best_acc:
            best_acc = vl_a
            best_state = copy.deepcopy(net.state_dict())

# Phase A
phase('Phase A — head only', epochs=10, lr=1e-3)

# Phase B — unfreeze блоков 5-7
for name, p in net.named_parameters():
    p.requires_grad = (
        name.startswith('features.5') or
        name.startswith('features.6') or
        name.startswith('features.7') or
        name.startswith('classifier.')
    )
trainable = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f'Phase B — fine-tune ({trainable:,} trainable params)')
phase('Phase B — fine-tune', epochs=8, lr=5e-5)

net.load_state_dict(best_state)
print(f'\nBest val accuracy: {best_acc:.4f}  ({best_acc*100:.1f}%)')

MODELS_OUT = WORKDIR / 'out'
MODELS_OUT.mkdir(exist_ok=True)
torch.save(best_state, MODELS_OUT / 'classifier.pth')

classes_meta = {
    '_comment': 'Regenerated by train_notebook.ipynb — PlantScope v3.x.',
    'model_version': '3.0.0',
    'architecture': {'detector': 'YOLOv8n', 'classifier': 'EfficientNetV2-S'},
    'num_classes': num_classes,
    'class_names': class_names,
    'detector_classes': ['leaf', 'diseased_leaf'],
    'val_accuracy': round(best_acc, 4),
}
(MODELS_OUT / 'classes.json').write_text(
    json.dumps(classes_meta, ensure_ascii=False, indent=2), encoding='utf-8')
print('classes.json →', MODELS_OUT / 'classes.json')

## 7. Тест пайплайна end-to-end

Прогоняем несколько валидационных картинок через обе модели: YOLOv8 находит bbox → кропаем регион → EfficientNetV2-S классифицирует.

In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F
from ultralytics import YOLO

det = YOLO(str(MODELS_OUT / 'detector.pt'))
net.eval()

val_images: list[tuple[Path, str]] = []
for cls in class_names:
    for img in (CLS_DIR / 'val' / cls).iterdir():
        val_images.append((img, cls))
random.shuffle(val_images)
demo_samples = val_images[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, (path, true_cls) in enumerate(demo_samples):
    ax = axes[i // 3][i % 3]
    img = Image.open(path).convert('RGB')
    arr = np.array(img)
    res = det.predict(arr, imgsz=640, conf=0.25, verbose=False)[0]

    ax.imshow(arr)
    pred_cls = 'not_a_plant'
    pred_conf = 0.0
    if res.boxes is not None and len(res.boxes):
        # рисуем все bbox; primary берём первый diseased, иначе первый
        boxes = res.boxes.xyxy.cpu().numpy()
        confs = res.boxes.conf.cpu().numpy()
        cls_ids = res.boxes.cls.cpu().numpy().astype(int)
        diseased_idx = [j for j, c in enumerate(cls_ids) if c == 1]
        primary = diseased_idx[0] if diseased_idx else 0
        for j, (x1, y1, x2, y2) in enumerate(boxes):
            color = 'red' if cls_ids[j] == 1 else 'lime'
            lw = 3 if j == primary else 1.5
            ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1,
                                       fill=False, edgecolor=color, linewidth=lw))
        x1, y1, x2, y2 = boxes[primary]
        crop = img.crop((int(x1), int(y1), int(x2), int(y2)))
        t = val_tf(crop).unsqueeze(0).to(device)
        with torch.no_grad():
            p = F.softmax(net(t), dim=1)[0].cpu().numpy()
        top = int(p.argmax())
        pred_cls = class_names[top]
        pred_conf = float(p[top])
    ok = pred_cls == true_cls
    ax.set_title(f'Pred: {pred_cls} ({pred_conf:.0%})\nTrue: {true_cls}',
                 color='green' if ok else 'red', fontsize=10)
    ax.axis('off')

plt.suptitle('End-to-end: YOLOv8 (bbox) + EfficientNetV2-S (class)', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Скачиваем веса

В Colab кнопка загрузки (либо монтируем Google Drive ниже). Положите файлы в `server/models/`:
- `detector.pt` — YOLOv8n
- `classifier.pth` — EfficientNetV2-S
- `classes.json` — список классов и версия модели

In [ ]:
import os
from pathlib import Path
try:
    from google.colab import files  # type: ignore
    colab = True
except ImportError:
    colab = False

det_size = os.path.getsize(MODELS_OUT / 'detector.pt') / 1e6
cls_size = os.path.getsize(MODELS_OUT / 'classifier.pth') / 1e6
print(f'detector.pt    — {det_size:.1f} MB')
print(f'classifier.pth — {cls_size:.1f} MB  (val_acc={best_acc:.1%})')

if colab:
    files.download(str(MODELS_OUT / 'detector.pt'))
    files.download(str(MODELS_OUT / 'classifier.pth'))
    files.download(str(MODELS_OUT / 'classes.json'))
else:
    print('Файлы лежат в', MODELS_OUT)

### (Опционально) Сохранить на Google Drive
Если не хочется качать через браузер — монтируем Drive.

In [ ]:
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    import shutil
    target = Path('/content/drive/MyDrive/PlantScope_v3')
    target.mkdir(parents=True, exist_ok=True)
    shutil.copy2(MODELS_OUT / 'detector.pt',    target / 'detector.pt')
    shutil.copy2(MODELS_OUT / 'classifier.pth', target / 'classifier.pth')
    shutil.copy2(MODELS_OUT / 'classes.json',   target / 'classes.json')
    print('Скопировано в', target)
except Exception as e:
    print('Пропущено:', e)